In [1]:
# Open the file in read mode
with open("/content/input.txt", "r") as file:
    content = file.read()
    print(content[:1100])


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [2]:
# Get the unique characters (vocabulary) in the text
chars = sorted(set(content))  # Unique characters in sorted order
vocab_size = len(chars)
# Print each character in the vocabulary
print("Characters in the dataset:")
print(chars)

# Print the vocabulary size
print(f"\nVocabulary size: {(vocab_size)}")

Characters in the dataset:
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

Vocabulary size: 65


In [4]:
string_to_index = {ch: i for i, ch in enumerate(chars)}  # String-to-Index
index_to_string = {i: ch for i, ch in enumerate(chars)}  # Index-to-String

# Step 4: Define encode and decode functions using lambdas
encode = lambda s: [string_to_index[c] for c in s]  # Encoder: string -> list of integers
decode = lambda l: ''.join([index_to_string[i] for i in l])  # Decoder: list of integers -> string

# Step 5: Input text to encode and decode
input_text = "good morning"

# Ensure all characters in input_text exist in the vocabulary
if all(char in string_to_index for char in input_text):
    encoded_text = encode(input_text)

    decoded_text = decode(encoded_text)

    print("Input Text:", input_text)
    print("Encoded Text:", encoded_text)
    print("Decoded Text:", decoded_text)
else:
    print("Error: Some characters in the input text are not in the vocabulary.")

Input Text: good morning
Encoded Text: [45, 53, 53, 42, 1, 51, 53, 56, 52, 47, 52, 45]
Decoded Text: good morning


In [5]:
import torch

data = torch.tensor(encode(content), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [6]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
print(train_data)
print(val_data)

tensor([47, 46, 43,  ...,  0, 58, 49])
tensor([60, 57, 16,  ..., 63, 58, 53])


In [7]:
train_data.shape, val_data.shape

(torch.Size([892315]), torch.Size([223079]))

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([47, 46, 43, 39,  6, 53, 45, 39,  1])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context.tolist()} the target is: {target.item()}")

When input is [47] the target is: 46
When input is [47, 46] the target is: 43
When input is [47, 46, 43] the target is: 39
When input is [47, 46, 43, 39] the target is: 6
When input is [47, 46, 43, 39, 6] the target is: 53
When input is [47, 46, 43, 39, 6, 53] the target is: 45
When input is [47, 46, 43, 39, 6, 53, 45] the target is: 39
When input is [47, 46, 43, 39, 6, 53, 45, 39] the target is: 1


In [10]:
torch.manual_seed(1337)
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class NgramLanguageModel(nn.Module):
    def __init__(self, vocab_size, block_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        self.block_size = block_size

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# nitialize the model and optimizer
model = NgramLanguageModel(vocab_size, block_size)

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [14]:
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [15]:
generated_text = decode(model.generate(torch.zeros((1, block_size), dtype=torch.long), max_new_tokens=500)[0].tolist())

print(generated_text)









y e  tWm,HAXj
xwE!r
 grOtr ofr lnmyjRd:r ETw::TIvriTlhYbtgcrgneYM.
INAm;b iybeh' wv.
 sAuDnbmd wh 
gw.u!ubfst
rbd hAtk
eIty,

twelpZYLst: rtGhea
LYatotsgiePNe'Pf'
Naaet,ii: aos nenZ

e  rh.thhrPh eh, so  Kn Io
wtemrp Fthiets-tersrbYyu dfm oc  br  rlgg o walhsnntl i aog R.iiZ'dLIskbymiseti PAraprNhtoUs oohod.dwoapU:bLMbls:teNf  lt
oThlTaWx;lfa
esh
ai.  vKo Tt fWbfIn;
 :daibsdtn,obnVlyedeEof
no Ftnig haI-t Serwh :bo nlad:
pevHiphSyNp
Bd coehea
 mnMgStLvTldZTVmhe  huomUws,i Gv ?Y'sCnlaM le,Ohlalc f
